# Modelo

Para ver como cambia la performance del modelo con el tratamiento de los datos primero hay que tener un modelo de partida que optimizar.

In [1]:
import numpy as np
import pandas as pd
import os
import joblib

from scipy.stats import randint, uniform, norm, loguniform

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, classification_report, confusion_matrix, roc_auc_score

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV

from transformers import AudioFilterResampler, SpectralSubtractor, MelSpectrogramTransformer, SpectrogramPadder, FlattenTransformer, FeatureExtractor

from config import TEST_SIZE, SEED

## Espectrogramas

Se suelen usar Mel espectrogramas

Traigo los consjuntos de entrenamiento y testeo. (TODO: buscar una manera más eficiente de guardarlos)

In [2]:
train_df = pd.read_csv('./dataset/ciclos/filtrado/train_melspectrogram.csv')
test_df = pd.read_csv('./dataset/ciclos/filtrado/test_melspectrogram.csv')

X_train = train_df.drop(columns=['label'])
y_train = train_df['label']
X_test = test_df.drop(columns=['label'])
y_test = test_df['label']

### Random Forest

#### Entrenamiento

In [11]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_distributions = {
    'max_depth': randint(10, 15),
    'min_samples_split': randint(5, 25),
    'min_samples_leaf': randint(5, 25)
}

Fine-tuning

El score elegido es auc-roc, podría ser recall o f1.

In [12]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END max_depth=13, min_samples_leaf=19, min_samples_split=15;, score=0.758 total time=   3.6s
[CV 2/5] END max_depth=13, min_samples_leaf=19, min_samples_split=15;, score=0.746 total time=   3.5s
[CV 3/5] END max_depth=13, min_samples_leaf=19, min_samples_split=15;, score=0.748 total time=   3.5s
[CV 4/5] END max_depth=13, min_samples_leaf=19, min_samples_split=15;, score=0.756 total time=   3.5s
[CV 5/5] END max_depth=13, min_samples_leaf=19, min_samples_split=15;, score=0.775 total time=   3.4s
[CV 1/5] END max_depth=14, min_samples_leaf=11, min_samples_split=23;, score=0.765 total time=   3.8s
[CV 2/5] END max_depth=14, min_samples_leaf=11, min_samples_split=23;, score=0.758 total time=   3.8s
[CV 3/5] END max_depth=14, min_samples_leaf=11, min_samples_split=23;, score=0.747 total time=   3.9s
[CV 4/5] END max_depth=14, min_samples_leaf=11, min_samples_split=23;, score=0.768 total time=   4.0s
[CV 5/5] END max_dep

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....00242B608F820>, 'min_samples_leaf': <scipy.stats....00242B223C5F0>, 'min_samples_split': <scipy.stats....00242B608F100>}"
,n_iter,20
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [13]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
16,13,6,10,0.770400
3,12,6,16,0.768028
4,11,5,16,0.766585
13,13,12,19,0.765789
1,14,11,23,0.764985
10,11,8,18,0.764096
19,11,8,18,0.764096
6,12,14,20,0.761439
9,12,11,13,0.761349
17,11,8,22,0.760926


In [14]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 13, 'min_samples_leaf': 6, 'min_samples_split': 10}
Best CV score: 0.7704003319553864


#### Evaluación

In [15]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))
print('AUC-ROC: ', round(roc_auc_score(y_test, y_pred), 2))
print('Recall: ', round(recall_score(y_test, y_pred), 2))

              precision    recall  f1-score   support

           0       0.72      0.57      0.64       429
           1       0.68      0.81      0.74       493

    accuracy                           0.70       922
   macro avg       0.70      0.69      0.69       922
weighted avg       0.70      0.70      0.69       922

AUC-ROC:  0.69
Recall:  0.81


#### Guardado

In [16]:
df_paths = pd.read_csv('./dataset/ciclos/filtrado/train.csv')
paths = ['./dataset/ciclos/' + f for f in df_paths['cycle_wav_file']]
y_train = df_paths['label'].values

In [17]:
# pipeline con la que se creo el dataset
pipeline_proc = Pipeline([
    ('filter_resample', AudioFilterResampler()),
    ('melspec', MelSpectrogramTransformer()),
    ('pad', SpectrogramPadder()),
    ('flatten', FlattenTransformer()),
    ('scaler', StandardScaler())
])

pipeline_final = Pipeline([
    ('preproc', pipeline_proc),
    ('rf', best_model)
])

pipeline_final.fit(paths, y_train)

,steps,"[('preproc', ...), ('rf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('filter_resample', ...), ('melspec', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,sr_target,16000
,highcut,4000
,lowcut,0


Guardo el modelo

In [19]:
joblib.dump(pipeline_final, './modelos/melspec_rf.pkl')

['./modelos/melspec_rf.pkl']

#### Evaluación en audios propios

Tengo grabaciones que fueron divididas en ciclos con el segmentador. El audio "respiración4_ciclo5" tiene una sibilancia clara, los demás ciclos de ese audio podrían ser positivos también pero no logro escuchar una sibilancia tan clara.

In [21]:
model = joblib.load('./modelos/melspec_rf.pkl')

In [ ]:
path = './audios_propios/ciclos_energy_based'
files = os.listdir(path)

df_audios_propios = pd.DataFrame(files, columns=['cycle_wav_file'])

df_audios_propios['label'] = np.where(df_audios_propios['cycle_wav_file'].str.contains('respiracion4_ciclo_5'), 1, 0)

df_audios_propios # audios de validacion

,cycle_wav_file,label
0,respiracion1_ciclo_1.wav,0
1,respiracion1_ciclo_2.wav,0
2,respiracion1_ciclo_3.wav,0
3,respiracion1_ciclo_4.wav,0
4,respiracion1_ciclo_5.wav,0
5,respiracion1_ciclo_6.wav,0
6,respiracion1_ciclo_7.wav,0
7,respiracion2_ciclo_1.wav,0
8,respiracion2_ciclo_2.wav,0
9,respiracion2_ciclo_3.wav,0


In [31]:
archivos = df_audios_propios['cycle_wav_file'].tolist()

X_propios_paths = [os.path.join(path, f) for f in archivos]
y_propios = df_audios_propios['label'].values

Predecimos

In [33]:
y_propios_pred = model.predict(X_propios_paths)
y_propios_proba_pred = model.predict_proba(X_propios_paths)[:, 1]

In [34]:
print(classification_report(y_propios, y_propios_pred))

              precision    recall  f1-score   support

           0       1.00      0.94      0.97        48
           1       0.25      1.00      0.40         1

    accuracy                           0.94        49
   macro avg       0.62      0.97      0.68        49
weighted avg       0.98      0.94      0.96        49



In [35]:
df_audios_propios['predicted'] = y_propios_pred
df_audios_propios['probability'] = y_propios_proba_pred.round(2)
df_audios_propios

,cycle_wav_file,label,predicted,probability
0,respiracion1_ciclo_1.wav,0,0,0.41
1,respiracion1_ciclo_2.wav,0,0,0.14
2,respiracion1_ciclo_3.wav,0,0,0.36
3,respiracion1_ciclo_4.wav,0,0,0.08
4,respiracion1_ciclo_5.wav,0,0,0.30
5,respiracion1_ciclo_6.wav,0,0,0.31
6,respiracion1_ciclo_7.wav,0,0,0.30
7,respiracion2_ciclo_1.wav,0,0,0.18
8,respiracion2_ciclo_2.wav,0,0,0.32
9,respiracion2_ciclo_3.wav,0,0,0.35


Parece prometedor, varios ciclos del audio 4 fueron clasificados como anormales.

## Atributos de Audio

1. Resample y Filtro pasa bajos
2. Feature Extractor
    + MFCC
    + ZCR
    + Short-Time Energy
    + SC
    + Spectral Roll-off
    + BER
    + Spectral Flatness
3. Scaler

In [36]:
train_df = pd.read_csv('./dataset/ciclos/filtrado/train_features.csv')
test_df = pd.read_csv('./dataset/ciclos/filtrado/test_features.csv')

X_train = train_df.drop(columns=['label'])
y_train = train_df['label']
X_test = test_df.drop(columns=['label'])
y_test = test_df['label']

### Random Forest

#### Entrenamiento

In [37]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_distributions = {
    'max_depth': randint(5, 15),
    'min_samples_split': randint(5, 25),
    'min_samples_leaf': randint(5, 25)
}

In [38]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END max_depth=11, min_samples_leaf=24, min_samples_split=19;, score=0.761 total time=   0.4s
[CV 2/5] END max_depth=11, min_samples_leaf=24, min_samples_split=19;, score=0.753 total time=   0.3s
[CV 3/5] END max_depth=11, min_samples_leaf=24, min_samples_split=19;, score=0.780 total time=   0.3s
[CV 4/5] END max_depth=11, min_samples_leaf=24, min_samples_split=19;, score=0.777 total time=   0.4s
[CV 5/5] END max_depth=11, min_samples_leaf=24, min_samples_split=19;, score=0.804 total time=   0.3s
[CV 1/5] END max_depth=12, min_samples_leaf=11, min_samples_split=23;, score=0.791 total time=   0.3s
[CV 2/5] END max_depth=12, min_samples_leaf=11, min_samples_split=23;, score=0.786 total time=   0.3s
[CV 3/5] END max_depth=12, min_samples_leaf=11, min_samples_split=23;, score=0.810 total time=   0.3s
[CV 4/5] END max_depth=12, min_samples_leaf=11, min_samples_split=23;, score=0.803 total time=   0.3s
[CV 5/5] END max_dep

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....00242654FD260>, 'min_samples_leaf': <scipy.stats....0024256600550>, 'min_samples_split': <scipy.stats....00242654FF8A0>}"
,n_iter,20
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [39]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
10,13,7,9,0.822435
4,12,7,6,0.818298
3,12,8,12,0.814077
18,12,8,6,0.814077
14,14,13,6,0.806131
1,12,11,23,0.803380
7,14,16,21,0.800697
5,12,16,10,0.795304
2,11,15,15,0.793739
13,13,18,22,0.790979


In [40]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 13, 'min_samples_leaf': 7, 'min_samples_split': 9}
Best CV score: 0.8224354783261132


#### Evaluación

In [41]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))
print('AUC-ROC: ', round(roc_auc_score(y_test, y_pred), 2))
print('Recall: ', round(recall_score(y_test, y_pred), 2))

              precision    recall  f1-score   support

           0       0.78      0.64      0.70       429
           1       0.73      0.84      0.78       493

    accuracy                           0.75       922
   macro avg       0.75      0.74      0.74       922
weighted avg       0.75      0.75      0.75       922

AUC-ROC:  0.74
Recall:  0.84


#### Guardado

In [42]:
df_paths = pd.read_csv('./dataset/ciclos/filtrado/train.csv')
paths = ['./dataset/ciclos/' + f for f in df_paths['cycle_wav_file']]
y_train = df_paths['label'].values

In [43]:
# pipeline con la que se creo el dataset
pipeline_feature_extraction = Pipeline([
    ('filter_resample', AudioFilterResampler()),
    ('feature_extractor', FeatureExtractor()),
    ('scaler', StandardScaler())
])

pipeline_final = Pipeline([
    ('feature_extraction', pipeline_feature_extraction),
    ('rf', best_model)
])

pipeline_final.fit(paths, y_train)

,steps,"[('feature_extraction', ...), ('rf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('filter_resample', ...), ('feature_extractor', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,sr_target,16000
,highcut,4000
,lowcut,0


Guardo el modelo

In [44]:
joblib.dump(pipeline_final, './modelos/features_rf.pkl')

['./modelos/features_rf.pkl']

#### Validación

In [45]:
modelo = joblib.load('./modelos/features_rf.pkl')

In [46]:
y_propios_pred = modelo.predict(X_propios_paths)
y_propios_proba_pred = modelo.predict_proba(X_propios_paths)[:, 1]

In [47]:
print(classification_report(y_propios, y_propios_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99        48
           1       0.00      0.00      0.00         1

    accuracy                           0.98        49
   macro avg       0.49      0.50      0.49        49
weighted avg       0.96      0.98      0.97        49



c:\Users\nazar\Clasificador-Respiracion\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nazar\Clasificador-Respiracion\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nazar\Clasificador-Respiracion\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

In [48]:
df_audios_propios['predicted'] = y_propios_pred
df_audios_propios['probability'] = y_propios_proba_pred.round(2)
df_audios_propios

,cycle_wav_file,label,predicted,probability
0,respiracion1_ciclo_1.wav,0,0,0.31
1,respiracion1_ciclo_2.wav,0,0,0.34
2,respiracion1_ciclo_3.wav,0,0,0.30
3,respiracion1_ciclo_4.wav,0,0,0.24
4,respiracion1_ciclo_5.wav,0,0,0.38
5,respiracion1_ciclo_6.wav,0,0,0.26
6,respiracion1_ciclo_7.wav,0,0,0.31
7,respiracion2_ciclo_1.wav,0,0,0.25
8,respiracion2_ciclo_2.wav,0,0,0.28
9,respiracion2_ciclo_3.wav,0,0,0.33
